# ✈️ Flight Delay Prediction — Data Cleaning

This notebook prepares the TunisAir training data for exploratory analysis and machine learning.

The source data comes from the **Zindi Flight Delay Prediction Challenge** and contains historical flight records from 2016–2018. The prediction target is `target`, representing flight delay in minutes.

### Main objectives

- inspect the raw dataset and identify data-quality issues
- standardize text fields and date/time variables
- account for airport time zones
- calculate scheduled flight duration
- create temporal and operational features
- enrich the data with airport-size information
- save one consistent cleaned dataset for the EDA and modeling notebooks

The output of this notebook is:

`data/Train_cleaned.csv`

This file becomes the input for the next stages of the project.


## 1. Initial Data Inspection

Before changing the data, we inspect its structure and the categorical variables.

This step helps us understand:

- the number of observations and variables
- the current data types
- the number of unique categories
- whether categorical values contain formatting inconsistencies
- which variables may require special treatment during cleaning

The raw dataset is kept unchanged. All transformations are later applied to a working copy called `df_clean`.


### Load the raw training data

The raw `Train.csv` file is loaded and its basic structure is inspected.

Keeping this first inspection close to the original data makes it easier to distinguish between issues already present in the source and changes introduced during cleaning.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
#from data_profiling import ProfileReport
import numpy as np
import datetime as dt

#Loading data
df = pd.read_csv('data/raw/Train.csv')

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
df.info()


In [ ]:
# Quick summary of categorical variables
categorical_columns = ["FLTID","DEPSTN","ARRSTN","STATUS","AC"]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts().head(5))

### Check text columns for whitespace

Categorical identifiers can look identical while actually containing leading or trailing spaces.

For example, `"TU 123"` and `"TU 123 "` would be treated as two different categories by a machine-learning model.

This cell identifies how many values in each text column contain surrounding whitespace before any cleaning is applied.


In [ ]:
# Finding leading/trailing whitespace in text columns
text_columns = df.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    original = df[column].astype(str)
    cleaned = original.str.strip()

    whitespace_rows = (original != cleaned).sum()

    print(f"{column}: {whitespace_rows} rows with leading/trailing spaces")

#print("Unique FLTID before cleaning:", df["FLTID"].nunique())

### Create the cleaned working dataframe

A separate `df_clean` dataframe is created so that the original `df` remains available for comparison.

The flight identifier `FLTID` is standardized by removing leading and trailing whitespace. This prevents duplicated categories caused only by formatting.


In [ ]:
df_clean = df.copy()

# Removing leading/trailing whitespace from text columns
df_clean["FLTID"] = df_clean["FLTID"].str.strip()
print("Unique FLTID after cleaning:", df_clean["FLTID"].nunique())

### Standardize flight and aircraft identifiers

Flight and aircraft identifiers should use a consistent text format before they are used for grouping, route analysis, or model encoding.

This step removes unnecessary whitespace from these identifiers.


In [ ]:
df_clean["FLTID"] = df["FLTID"].str.strip()
df["FLTID"] = df["FLTID"].replace(" ", "", regex=True)
df["AC"] = df["AC"].replace(" ", "", regex=True)

## 2. Date and Time Preparation

The source dataset stores dates and scheduled times as strings.

They are converted to pandas datetime objects so that we can:

- sort flights chronologically
- extract calendar features
- compare scheduled departure and arrival times
- calculate flight duration
- create aircraft rotation and interval features

`STD` and `STA` use different separators in the raw source, so they require separate parsing formats.


In [ ]:
df_clean["DATOP"] = pd.to_datetime(df_clean["DATOP"],format="%Y-%m-%d")
#df_clean["DATOP"].head()
#df_clean["DATOP"].dtype

df_clean["STD"] = pd.to_datetime(df_clean["STD"],format="%Y-%m-%d %H:%M:%S") # STD using : as separator
df_clean["STD"].dtype

#STA using . as separator, this line will convert it to datetime format same as STD and DATOP
df_clean["STA"] = pd.to_datetime(df_clean["STA"], format="%Y-%m-%d %H.%M.%S") 
df_clean["STA"].dtype

## 3. Airport Time Zones

Scheduled departure and arrival times are recorded in local airport time.

A direct subtraction of local times can therefore produce incorrect flight durations when a flight crosses time zones.

To make scheduled times comparable, each departure and arrival airport is mapped to its IANA time zone using the `airportsdata` package.


### Map departure and arrival airports to time zones

The IATA airport code is used to retrieve the appropriate time zone for each airport.

We explicitly check for airports that cannot be found in the current airport database. Historical airport codes may be missing because airport databases reflect current infrastructure.

`SXF` is handled manually because it is a historical Berlin airport code that is not available in the current lookup.


In [ ]:
import airportsdata
from zoneinfo import ZoneInfo
import datetime as dt

# Load airport data by IATA code
airports = airportsdata.load("IATA")

# Add timezone based on departure and arrival airport
df_clean["DEP_timezone"] = df_clean["DEPSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

df_clean["ARR_timezone"] = df_clean["ARRSTN"].map(
    lambda code: airports.get(code, {}).get("tz"))

#print(df_clean["DEP_timezone"].isna().sum())
#print(df_clean["ARR_timezone"].isna().sum())

print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())

# Historical airport (SXF) not found in current airportsdata
df_clean.loc[df_clean["DEPSTN"] == "SXF", "DEP_timezone"] = "Europe/Berlin"
df_clean.loc[df_clean["ARRSTN"] == "SXF", "ARR_timezone"] = "Europe/Berlin"

df_clean.columns

### Validate the time-zone mapping

After adding the manual historical-airport mapping, the dataset is checked again for missing departure or arrival time zones.

This validation is important because missing time zones would prevent the UTC conversion and scheduled-duration calculation in the next steps.


In [ ]:
print(df_clean.loc[df_clean["DEP_timezone"].isna(),"DEPSTN"].unique())
print(df_clean.loc[df_clean["ARR_timezone"].isna(),"ARRSTN"].unique())
df_clean.info()


## 4. Convert Scheduled Times to UTC

To compare scheduled departure and arrival times consistently across countries, local airport times are converted to UTC.

The conversion uses:

- `STD` + departure airport time zone → `STD_UTC`
- `STA` + arrival airport time zone → `STA_UTC`

UTC timestamps provide a common reference independent of the local time zone.


### UTC conversion helper

The `to_utc()` function combines each local scheduled datetime with its airport time zone and converts it to UTC.

Missing datetime or time-zone information is returned as `NaT` rather than forcing an invalid conversion.


In [ ]:
def to_utc(dt, tz):
    if pd.isna(dt) or pd.isna(tz):
        return pd.NaT
    
    return (
        dt.replace(tzinfo=ZoneInfo(tz))
          .astimezone(ZoneInfo("UTC"))
    )


df_clean["STD_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STD"],
        df_clean["DEP_timezone"]
    )
]

df_clean["STA_UTC"] = [
    to_utc(dt, tz)
    for dt, tz in zip(
        df_clean["STA"],
        df_clean["ARR_timezone"]
    )
]



## 5. Scheduled Flight Duration

Once both scheduled timestamps are expressed in UTC, the scheduled flight duration can be calculated reliably.

The resulting `duration` feature represents the number of minutes between scheduled departure and scheduled arrival.

This feature can help the models distinguish short-, medium-, and long-haul operations.


### Calculate and validate `duration`

Flight duration is calculated in minutes using the difference between `STA_UTC` and `STD_UTC`.

Afterward, missing duration values are counted. Missing values at this stage would generally indicate that one of the required scheduled timestamps or time zones could not be resolved.


In [ ]:
df_clean["duration"] = (df_clean["STA_UTC"] - df_clean["STD_UTC"]).dt.total_seconds() / 60

print(df_clean["duration"].isna().sum())

## 6. Calendar Features

Flight-delay behavior may vary with the weekly and seasonal operating schedule.

We therefore derive several calendar features from the scheduled departure time:

- `weekday`
- `weekend`
- `season`
- `hour`
- `month`
- `year`

These features allow the EDA and models to capture recurring temporal patterns.


### Weekday, weekend and season

`weekday` represents the numeric day of the week.

The project also creates a custom `weekend` indicator based on the operational grouping used in the analysis.

`season` groups months into Winter, Spring, Summer and Autumn, providing a broader representation of annual patterns than month alone.


In [ ]:
df_clean["weekday"] = df_clean["STD_UTC"].dt.dayofweek
df_clean["weekend"] = df_clean["weekday"].map({0:True, 1:False, 2:False, 3:False, 4:True, 5:True, 6:True})
df_clean["season"] = df_clean["STD_UTC"].dt.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})


In [ ]:
df_clean[df_clean["STA"].dt.year == 2019].shape[0] #Removes flights with Arrival times in 2019
df_clean['hour'] = df_clean["STD_UTC"].dt.hour
df_clean['weekday'] = df_clean["STD_UTC"].dt.weekday
df_clean['month'] = df_clean["STD_UTC"].dt.month
df_clean['year'] = df_clean["STD_UTC"].dt.year

## 7. Route Feature

Departure and arrival airports are combined into a single directional `route` variable.

For example:

`TUN_CDG`

and

`CDG_TUN`

remain different routes.

This allows the EDA and machine-learning models to capture route-specific operating patterns.


### Build the directional route identifier

`DEPSTN` and `ARRSTN` are concatenated into one categorical feature called `route`.

The direction is intentionally preserved because operating conditions can differ between the outbound and return directions.


In [ ]:
df_clean["route"] = (df_clean["DEPSTN"] + "_" + df_clean["ARRSTN"])

## 8. Aircraft Rotation Sequence

Flights operated under the same aircraft code on the same day are ordered and assigned a sequence number.

`rotation_sequence = 1` represents the first recorded flight for that aircraft code on that operating date, `2` the second, and so on.

This feature is an operational proxy that can help investigate whether delays tend to change as the daily flight sequence progresses.

> Note: `AC` is an aircraft code in the challenge data. It should not automatically be interpreted as a verified unique physical airframe.


### Calculate `rotation_sequence`

The data is sorted by operating date and aircraft code. `cumcount()` then numbers the observations within every `DATOP`–`AC` group.

This produces a simple aircraft-day workload/sequence feature without using the target variable.


In [ ]:
df_clean = df_clean.sort_values(["DATOP","AC"]).copy()

df_clean["rotation_sequence"] = (df_clean.groupby(["DATOP", "AC"]).cumcount() + 1)

df_clean.head(10)

## 9. Interval Between Consecutive Flights

A second aircraft-related feature measures the scheduled time between the previous recorded arrival and the next scheduled departure for the same aircraft code.

The previous scheduled arrival is stored in `previous_STA_UTC`, and the difference is stored in `interval` in minutes.

Short intervals may represent tighter operating schedules, while larger intervals provide more recovery time.

The first recorded flight for each aircraft code naturally has no previous arrival and therefore receives a missing interval.


In [ ]:
# Interval between previous flight and next flight
df_clean = (
    df_clean
    .sort_values(["AC", "STD_UTC"])
    .copy()
)

df_clean["previous_STA_UTC"] = (
    df_clean
    .groupby("AC")["STA_UTC"]
    .shift(1)
)
df_clean["interval"] = (
    df_clean["STD_UTC"] - df_clean["previous_STA_UTC"]
).dt.total_seconds() / 60

## 10. Airport Size Enrichment

The EDA indicated that departure and arrival airports are important for delay behavior.

To provide the model with additional context, the flight data is enriched with an external airport-volume dataset. `TotalSeats` is used as a proxy for airport size/activity.

Two features are created:

- `DEPSIZE` — size proxy for the departure airport
- `ARRSIZE` — size proxy for the arrival airport

A few small airports are not available in the external dataset. They are assigned the 25th percentile of observed airport seat volume as a conservative approximation.

This external information is intended to represent differences in airport scale and potential operational complexity.


### Load airport-volume data

The airport-volume reference file is reduced to the airport code (`Orig`) and `TotalSeats`.

Missing airports identified in the project are added before merging so that the final training data has an airport-size value wherever possible.


In [ ]:
# downloaded Airport size data from https://datacatalog.worldbank.org/search/dataset/0038117/global-airports
airports = pd.read_csv("data/airport_volume_airport_locations.csv")
airports = airports[["Orig", "TotalSeats"]]

# Three airports are missing from this database: 'BYJ', 'GAF', 'EBM'
# These are small airports, so we will fill them with the minimum of the other airports
missing_airports = pd.DataFrame({
    "Orig": ["BYJ", "GAF", "EBM"],
    "TotalSeats": [airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25), 
                   airports["TotalSeats"].quantile(0.25)]})
airports = pd.concat([airports, missing_airports], ignore_index=True)

df_clean = pd.merge(df_clean, airports, how="left", left_on="DEPSTN", right_on="Orig")
df_clean.rename(columns={'TotalSeats': 'DEPSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)



In [ ]:
df_clean = pd.merge(df_clean, airports, how="left", left_on="ARRSTN", right_on="Orig", suffixes=('_y', '_x'))
df_clean.rename(columns={'TotalSeats': 'ARRSIZE'}, inplace=True)
df_clean.drop(columns=["Orig"], inplace=True)
df_clean.head(5)

## 11. Save the Modeling Dataset

At this point the raw flight information has been standardized and enriched with temporal, route, aircraft-sequence, interval, time-zone, duration, and airport-size features.

The cleaned dataframe is now ready to be used consistently by:

- `02_eda.ipynb`
- `03_models.ipynb`
- `04_prediction_test.ipynb`

Keeping the cleaned dataset as a separate file makes the workflow reproducible and prevents the EDA/model notebooks from repeating the cleaning process.


### Export `Train_cleaned.csv`

The final dataframe is saved to `data/processed/Train_cleaned.csv`.

The prediction notebook must reproduce the **same feature-engineering logic** on `Test.csv` before sending test observations to the trained model.


In [ ]:
df_clean.to_csv("data/Train_cleaned.csv", index=False)